## Implement Transformer
Transformer Architecture 구현 코드

참고한 코드 출처: https://cpm0722.github.io/pytorch-implementation/transformer

위 블로그의 원본 구현을 참고하여 수정하였습니다. 

In [21]:
import torch.nn as nn
import numpy as np

class Transformer(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def encode(self, x, src_mask):
        out = self.encoder(x, src_mask)
        return out
        
    def decode(self, z, c, tgt_mask):
        """ c: context, z: sentence"""
        out = self.decoder(x, c, tgt_mask)
        return out

    def forward(self, x, z, mask):
        # masking
        src_mask = self.make_src_mask(x)
        tgt_mask = self.make_tgt_mask(z)

        # encoder & decoder
        c = self.encode(x, src_mask) # encoder는 context를 생성
        y = self.decoder(z, c, tgt_mask) # decoder는 context를 사용
        return y
        
    def make_pad_mask(self, Q, K, pad_idx = 1):
        """
        embedding 획득하기 전, token sequence 상태로 들어오는 것.
        Q: (n_batch, Q_n)
        K: (n_batch, K_n)
        pad_idx : 대개 1이다. 이와 일치하는 token들은 0, 그 외에는 모두 1인 mask를 생성
        """
        Q_n, K_n = Q.size(1), K.size(1)
    
        K_mask = K.ne(pad_idx).unsqueeze(1).unsqueeze(2)  # (n_batch, 1, 1, K_n)
        K_mask = K_mask.repeat(1, 1, Q_n, 1)  # (n_batch, 1, Q_n, K_n) # Q_n번 만큼 복제 (1이면 변경 없는 의미)
    
        Q_mask = Q.ne(pad_idx).unsqueeze(1).unsqueeze(3)  # (n_batch, 1, Q_n, 1)
        Q_mask = Q_mask.repeat(1, 1, 1, K_n)  # (n_batch, 1, Q_n, K_n) # Q_n번 만큼 반복 (1이면 변경 없는 의미)
    
        mask = Q_mask & Q_mask
        mask.requires_grad = False
        return mask
        
    def make_src_mask(self, src):
        """Cross Attention일 때 서로 다른 값이 들어올 수 있어서"""
        pad_mask = self.make_pad_mask(src, src)
        return pad_mask

    
    def mask_subsequent_mask(self, Q, K):
        """
        i번째 토큰 생성 시 1~ i-1번째 까지만 보이게 하는 함수
        Q: (n_batch, Q_n)
        K: (n_batch, K_n)
        """
        Q_n, K_n = Q.size(1), K.size(1)
    
         # lower triangle without diagonal
        tril = np.tril(np.ones((query_seq_len, key_seq_len)), k=0).astype('uint8') # (Q,K)
        mask = torch.tensor(tril, dtype=torch.bool, requires_grad=False, device=query.device)
        return mask
        
    def make_tgt_mask(self, tgt):
        """
        Decoder에선 subsequent + pad mask 를 구해야함
        """
        pad_mask = self.make_pad_mask(tgt, tgt)
        seq_mask = self.make_subsequent_mask(tgt, tgt)
        mask = pad_mask & seq_mask
        return pad_mask & seq_mask
        
    def make_src_tgt_mask(self, src, tgt):
        """
        Decoder의 Cross Attention부분에서 사용하는 mask 함수
        Self-Head-Multi-Head Attention Layer에서 넘어온 Q
        Encoder에서 넘어온 K, V
        """
        pad_mask = self.make_pad_mask(tgt, src)
        return pad_mask

In [3]:
import copy
class Encoder(nn.Module):
    def __init__(self, encoder_block, n_layer):
        super().__init__()
        self.layers = [copy.deepcopy(encoder_block) for _ in range(n_layer)]
        # self.layers = nn.Sequential(*[copy.deepcopy(encoder_block) for _ in range(n_layer)])

    def forward(self, x, mask):
        out = x
        for layer in self.layers:
            out = layer(out, mask)
        return out

In [4]:
class EncoderBlock(nn.Module):
    def __init__(self, self_attention, position_ff):
        super().__init__()
        self.self_attention = self_attention # Multi-Head Attention Layer
        self.position_ff = position_ff # Position-wise Feed-Forward Layer

        self.residuals = [ResidualConnectionLayer() for _ in range(2)]
        
    def forward(self, x, mask):
        # lambda를 사용하는 이유는 residual에서 sublayer가 실행되기 때문이다. -> 더 간단히 residual에서 sublayer(x)로 실행하기 때문에.
        # 즉시 실행되지 않고, sublayer(x)가 호출 될 때 실행된다.
        out = x
        out = self.residuals[0](out, lambda out: self.self_attention(query=out, key=out, value=out, mask=mask))
        out = self.residuals[1](out, self.position_ff)
        return out

###  Scaled Dot-Product Attention 계산 Flow

#### Attention Score
1. Q, K, V Embedding ($n$X$d_\text{embed}$)
2. Q, K, V FC Layer (($n$ X $d_\text{embed}$ ) X ($d_\text{embed}$ X $d_k$)  =  ($n$ X $d_k$) )
3. Q, $K^T$ Matmul (($n$ X $d_k$) X ($d_k$ X $n$) = ($n$ X $n$))) -> `Attention Score`
#### Final Result
4. Attention Score Scale
5. Mask
6. Softmax
7. `6 (Softmax)`,V Matmul (($n$ X $n$) X ($n$ X $d_k$) = ($n$ X $d_k$)))

- 실제 구현 시 mini-batch로 인하여 n_batch가 추가된다

In [5]:
import torch
import math
import torch.nn.functional as F

def calc_attention(Q, K, V, mask):
    """
    self-attention 계산하는 함수( 3번 ~ 7번)
    Q,K,V fc층을 지난 값이 통과된다.
    Q, K, V: (n_batch, h, n, d_k)
    mask: (n_batch, 1, n, n)
    """
    d_k = Q.shape[-1]
    attention_score = torch.matmul(Q, K.transpose(-2,-1)) # 3번 (n, h, n, n)
    attention_score_scale = attention_score /  math.sqrt(d_k) # 4번
    if mask: # mask가 있으면
        attention_score_scale = attention_score_scale.masked_fill(mask == 0, -1e9) # 5번
        
    attention_prob = F.softmax(attention_score_scale, dim = -1) # 6번  (n, h, n, n)
    out = torch.matmul(attention_prob, V) # 7번 (n, h, n, d_k)
    return out

### Multi-Head Attention Layer 계산 
- Attention을 병렬적으로 수행하기 위한 방법
- 더 많은 attention들을 포함할 수 있음
    - 서로 다른 위치에서 다양한 표현 하위 공간의 정보를 동시에 주목 가능

#### 일반적인 Multi-Head Attention 계산 방식
- Transformer 논문에서는 $h=8$ 개의 Attention Head를 사용하였다.
- $h=8$ 개의 Head를 병렬적으로 계산하는 경우, 기존의 Query(Q), Key(K), Value(V)를 생성하기 위해 각각 3개의 FC Layer가 필요하다.
- 따라서, Multi-Head Attention에서는 총 $3*h$개의 FC Layer가 필요하게 된다.
- 각 FC Layer의 최종 출력은 $(n \times d_k)$ 형태를 가지며, 이를 $h$개의 Head에 대해 병렬적으로 계산한 후 Concatenation 하면 $(n \times (d_k * h))$ 형태의 행렬이 생성된다.
- 즉, Q, K, V 각각에 대해 독립적으로 $h$번의 Attention 연산을 수행해야 하므로 **계산량이 증가하며, 비효율적이다.**

#### 제안된 Multi-Head Attention 계산 방식
- 실제로는 Q, K, V 자체를 $(n \times d_\text{model}) (d_\text{model} = d_k * h)$ 로 생성하여 **한 번의 Self-Attention 계산으로 output을 만들게 된다.**
    - $d_\text{model}$과 $d_\text{embed}$랑 같다고 생각하면 된다.
- 단순하게 생각하면 $d_k$를 $d_\text{model}$로 변경하는 것이다.

In [6]:
class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, d_model, h, qkv_fc, out_fc):
        """
        qkv_fc: (d_embed, d_model)
        out_fc: (d_model, d_embed) -> attention 계산 이후 거쳐가는 FC Layer.
        """
        super().__init__()

        self.d_model = d_model
        self.h = h
        self.q_fc = copy.deepcopy(qkv_fc) # 서로 다른 layer를 위하여 deepcopy
        self.k_fc = copy.deepcopy(qkv_fc)
        self.v_fc = copy.deepcopy(qkv_fc)
        self.out_fc = out_fc

    def forward(self, *args, Q, K, V, mask=None):
        """
        Q, K, V: (n_batch, n, d_embed)
        mask: (n_batch, n, n) -> 한 문장에 대한 것
        return value: (n_batch, h, n, d_k)

        Q, K, V를 따로 인자를 받는 이유는 Decoder에서 Cross-Attention을 사용하기 때문이다.
        """
        n_batch = Q.size(0)

        def transform(x, fc): # (n_batch, n, d_embed)
            """ Q, K, V를 구하는 함수 """
            out = fc(x) # (n_batch, n, d_model)
            out = out.view(n_batch, -1, self.h, self.d_model // self.h) # (n_batch, n, h, d_k)
            out = out.transpose(1, 2) # (n_batch, h, n, d_k) # calc_attention의 input shape에 맞추기 위하여
            return out
            
        # Q, K, V -> FC 층 (2번)
        Q = transform(Q, self.q_fc) # (n_batch, h, n, d_k)
        K = transform(K, self.k_fc) # (n_batch, h, n, d_k)
        V = transform(V, self.v_fc) # (n_batch, h, n, d_k)

        # 3번 ~ 7번
        out = self.calc_attention(Q, K, V, mask) # (n_batch, h, n, d_k)

        out = out.transpose(1, 2) # (n_batch, n, h, d_k)
        out = out.contiguous().view(n_batch, -1, self.d_model) # (n_batch, n, d_model)
        
        out = self.out_fc(out) # (n_batch, n, d_embed)
        return out

In [8]:
class PositionWiseFeedForwardLayer(nn.Module):
    """ 
    Multi-Head Attention Layer의 output을 input으로 받아 연산을 수행
    다음 Encoder Block에게 output을 넘겨주는 역할
    """
    def __init__(self, fc1, fc2):
        super().__init__()
        self.fc1 = fc1   # (d_embed, d_ff)
        self.relu = nn.ReLU()
        self.fc2 = fc2 # (d_ff, d_embed)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

In [9]:
class ResidualConnectionLayer(nn.Module):
    """
    잔차 연결로 인하여 gradient vanishing 현상 방지 가능
    """
    def __init__(self):
        super().__init__()
        pass
    def forward(self, x, sub_layer):
        out = x
        out = sub_layer(out)
        return out + x

### Decoder

- `Teacher Forcing`: 실제 labeled data(Ground Truth)를 RNN cell의 input으로 사용하는 것이다. 
    - 그 이유는 모델에 계속 sequence하게 output이 input으로 들어가게 되는데, 이 때 초기 훈련 시 한 번 token이 잘못 생성 되면 훈련 자체가 엉터리가 될 수 있기 때문이다.
      - 정확히는 Ground Truth의 [:-1]로 slicing을 한 것이다(마지막 token인 EOS token을 제외하는 것이다)

- Transformer에서 `Teacher Forcing`을 구현하기 위하여 단순히 slicing만 하면 안된다.
- 왜냐하면 병렬 연산을 해야하기 때문에 ground truth의 embedding을 matrix로 만들어 input으로 그대로 사용하게 되면,Decoder에서 Self-Attention 연산을 수행하게 될 때 현재 출력해내야 하는 token의 정답까지 알고 있는 상황이 발생
- 따라서 Masking이 필요. i 번째 토큰 생성할 때 `1~ i-1`의 토큰만 보이게 하는 것.


```python
np.tril(np.ones((10, 10)), k=0)

# output: 
array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 1., 1., 0., 0., 0., 0., 0., 0., 0.],
       [1., 1., 1., 1., 0., 0., 0., 0., 0., 0.],
       [1., 1., 1., 1., 1., 0., 0., 0., 0., 0.],
       [1., 1., 1., 1., 1., 1., 0., 0., 0., 0.],
       [1., 1., 1., 1., 1., 1., 1., 0., 0., 0.],
       [1., 1., 1., 1., 1., 1., 1., 1., 0., 0.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 0.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])
```

In [10]:
import numpy as np
def mask_subsequent_mask(Q, K):
    """
    i번째 토큰 생성 시 1~ i-1번째 까지만 보이게 하는 함수
    Q: (n_batch, Q_n)
    K: (n_batch, K_n)
    """
    Q_n, K_n = Q.size(1), K.size(1)

     # lower triangle without diagonal
    tril = np.tril(np.ones((query_seq_len, key_seq_len)), k=0).astype('uint8') # (Q,K)
    mask = torch.tensor(tril, dtype=torch.bool, requires_grad=False, device=query.device)
    return mask

## Implement Decoder

- Decoder 역시 Encoder와 마찬가지로 $N$개의 Decoder Block이 겹겹이 쌓인 구조.
- Decoder에서 주의해야 할 점은 Encoder에서 넘어오는 context가 Decoder에 들어간다는 것.
  - 정확히는 Decoder Block의 `Cross-Multi-Head Attention Layer`에서 사용한다.
  - Encoder에서 넘어온 context를 key, value로 사용하고 Decoder의 Self-Multi-Head Attention에서 넘어온 것을 Query로 사용한다. (그래서 cross attention이다.)
    - Decoder에서 도출해내고자 하는 최종 output은 teacher forcing으로 넘어온 sentence와 **최대한 유사한 predicted sentence이다.** 그래서 Query가 self-attention의 Output이다.

In [22]:
class Decoder(nn.Module):

    def __init__(self, decoder_block, n_layer):
        super().__init__()

        self.n_layer = n_layer
        self.layers = nn.ModuleList([copy.deepcopy(decoder_block) for _ in range(self.n_layer)])
        
    def forward(self, tgt, encoder_out, tgt_mask, src_tgt_mask):
        """
        tgt_mask: Decoder input으로 주어지는 taget sentence의 pad masking과 subsequent masking
        src_tgt_mask: 
        """
        out = tgt
        for layer in self.layers:
            out = layer(out, encoder_out, tgt_mask, src_tgt_mask)
        return out